# LLaVA-1.5-7B Healthcare Confidence Audit: Clean Run

This notebook is a clean replacement for the earlier debugging notebook.

**Important rules**
1. Enable a **Tesla T4 GPU** and turn **Internet on**.
2. Run Cell 1 once. Restart the session only if Cell 1 tells you to.
3. After reconnection, start from Cell 2.
4. Run the model-loading cell only once per session. It contains a guard against accidental double-loading.
5. Start with the 32-evaluation pilot. Do not increase sample sizes until the pilot finishes.

Research only. Not for diagnosis, triage, or clinical deployment.


In [ ]:
# CELL 1 — INSTALL ONLY IF REQUIRED
# Run once. If packages are installed or upgraded, this cell restarts the kernel.

import os, sys, subprocess, importlib.metadata as md

def version_tuple(v):
    parts=[]
    for value in str(v).split(".")[:3]:
        digits="".join(ch for ch in value if ch.isdigit())
        parts.append(int(digits or 0))
    return tuple((parts+[0,0,0])[:3])

def installed(name):
    try:
        return version_tuple(md.version(name))
    except md.PackageNotFoundError:
        return (0,0,0)

requirements={
    "bitsandbytes": (0,46,1),
    "accelerate": (1,2,0),
    "transformers": (4,48,0),
    "datasets": (3,0,0),
}

needs_install=any(installed(name)<minimum for name,minimum in requirements.items())

if needs_install:
    print("Installing required packages. The kernel will restart once.")
    subprocess.check_call([
        sys.executable,"-m","pip","install","-q","--no-cache-dir","--upgrade",
        "bitsandbytes>=0.46.1,<0.51",
        "accelerate>=1.2,<2",
        "transformers>=4.48,<4.58",
        "datasets>=3,<5",
        "sentencepiece>=0.2,<0.3",
        "packaging>=24,<26",
        "psutil>=5.9,<8",
    ])
    print("Installation complete. Restarting now. After reconnection, run Cell 2.")
    os.kill(os.getpid(), 9)
else:
    print("Required packages already exist. Continue to Cell 2. Do not restart.")


Installing required packages. The kernel will restart once.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 246.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 341.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 229.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 342.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 270.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 288.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 345.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 349.3 MB/s eta 0:00:00


In [ ]:
# CELL 2 — IMPORTS, CONFIGURATION, AND CLEAN GPU CHECK

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

import re, ast, gc, json, random, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import bitsandbytes as bnb
import psutil
from PIL import ImageFilter, ImageEnhance
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from transformers.utils import is_bitsandbytes_available

warnings.filterwarnings("ignore")

OUT="/kaggle/working/healthcare_audit_clean" if os.path.exists("/kaggle/working") else "/content/healthcare_audit_clean"
CFG={
    "model":"llava-hf/llava-1.5-7b-hf",
    "seed":42,
    "n_slake":20,
    "n_pathvqa":20,
    "n_afrimed":40,
    "conditions":["clean","blur","dark"],
    "checkpoint_every":1,
    "ece_bins":8,
    "high_confidence":0.80,
    "output_dir":OUT,
    "resume":True,
}
SEED=CFG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
os.makedirs(OUT,exist_ok=True)
PRED_PATH=os.path.join(OUT,"predictions.csv")
ERROR_PATH=os.path.join(OUT,"evaluation_errors.csv")

assert torch.cuda.is_available(), "Enable the Kaggle Tesla T4 GPU."
assert is_bitsandbytes_available(), "Transformers cannot detect bitsandbytes. Rerun Cell 1 and allow the restart."

gc.collect(); torch.cuda.empty_cache()
allocated=torch.cuda.memory_allocated()/1024**3
reserved=torch.cuda.memory_reserved()/1024**3
print("GPU:",torch.cuda.get_device_name(0))
print("bitsandbytes:",bnb.__version__)
print("Allocated GPU GB:",round(allocated,2))
print("Reserved GPU GB:",round(reserved,2))
print("Output:",OUT)
if allocated>0.5 or reserved>0.5:
    raise RuntimeError("GPU is not clean. Use Run > Restart Session, then begin again from Cell 2.")
print("Cell 2 complete. GPU is clean.")


In [ ]:
# CELL 3 — LOAD LLAVA EXACTLY ONCE

if "model" in globals() and model is not None:
    raise RuntimeError("LLaVA is already loaded. Do not run Cell 3 twice in one session.")

MODEL_NAME=CFG["model"]
print("Loading processor...")
processor=AutoProcessor.from_pretrained(MODEL_NAME,use_fast=False)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token=processor.tokenizer.eos_token
processor.tokenizer.padding_side="left"

quantization_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading LLaVA-1.5-7B in 4-bit mode...")
model=LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"":0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.eval()
for parameter in model.parameters(): parameter.requires_grad=False

print("Model device:",next(model.parameters()).device)
print("Evaluation mode:",not model.training)
print("Allocated GPU GB:",round(torch.cuda.memory_allocated()/1024**3,2))
print("Reserved GPU GB:",round(torch.cuda.memory_reserved()/1024**3,2))
print("Cell 3 complete. NEVER run Cell 3 again in this session.")


In [ ]:
# CELL 4 — SCORING FUNCTIONS AND ONE INTERNAL TOKEN CHECK

def option_token_id(letter):
    for candidate in [" "+letter,letter]:
        ids=processor.tokenizer.encode(candidate,add_special_tokens=False)
        if len(ids)==1: return ids[0]
    raise ValueError(f"Option {letter} is not one token")

def predict_options(question,options,image=None):
    letters=[chr(65+i) for i in range(len(options))]
    choices="\n".join(f"{letter}. {option}" for letter,option in zip(letters,options))
    if image is None:
        prompt=("USER:\nResearch-only healthcare benchmark. Select the best answer. "
                "Return only the option letter.\n"+question+"\n"+choices+"\nASSISTANT: ")
        inputs=processor(text=prompt,return_tensors="pt").to(model.device)
    else:
        prompt=("USER: <image>\nResearch-only medical image benchmark. Select the best answer using the image. "
                "Choose insufficient evidence if needed. Return only the option letter.\n"+
                question+"\n"+choices+"\nASSISTANT: ")
        inputs=processor(text=prompt,images=image,return_tensors="pt").to(model.device)
    with torch.inference_mode():
        logits=model(**inputs).logits[0,-1].float()
    ids=torch.tensor([option_token_id(x) for x in letters],device=logits.device)
    probabilities=torch.softmax(logits[ids],dim=0)
    if not torch.isfinite(probabilities).all():
        raise FloatingPointError("Non-finite option probabilities")
    probabilities=probabilities.detach().cpu().numpy().astype(float)
    prediction=int(probabilities.argmax())
    return prediction,float(probabilities[prediction]),probabilities.tolist()

def perturb_image(image,condition):
    image=image.convert("RGB")
    if condition=="clean": return image
    if condition=="blur": return image.filter(ImageFilter.GaussianBlur(2.2))
    if condition=="dark": return ImageEnhance.Brightness(image).enhance(0.45)
    raise ValueError(condition)

print("Option token IDs:",{x:option_token_id(x) for x in ["A","B","C","D"]})
print("Cell 4 complete.")


In [ ]:
# CELL 5 — LOAD SMALL STREAMING VISUAL SAMPLES

def balanced_stream_sample(dataset,n,name,domain):
    target=max(1,n//2); yes=[]; no=[]; examined=0
    for example in dataset:
        examined+=1
        answer=str(example.get("answer","")).strip().lower()
        question=str(example.get("question","")).strip()
        image=example.get("image")
        if answer not in {"yes","no"} or not question or image is None: continue
        row={"dataset":name,"domain":domain,"image":image,"question":question,
             "answer":answer,"country":"not specified","african_origin":False}
        if answer=="yes" and len(yes)<target: yes.append(row)
        elif answer=="no" and len(no)<target: no.append(row)
        if len(yes)>=target and len(no)>=target: break
    frame=pd.DataFrame(yes+no)
    if len(frame)<2*target: raise ValueError(f"Could not collect balanced {name} sample")
    print(f"{name}: examined {examined}, selected {len(frame)}")
    return frame.sample(frac=1,random_state=SEED).reset_index(drop=True)

print("Loading SLAKE streaming sample...")
slake_frame=balanced_stream_sample(
    load_dataset("mdwiratathya/SLAKE-vqa-english",split="test",streaming=True),
    CFG["n_slake"],"SLAKE","radiology")

print("Loading PathVQA streaming sample...")
pathvqa_frame=balanced_stream_sample(
    load_dataset("flaviagiammarino/path-vqa",split="test",streaming=True),
    CFG["n_pathvqa"],"PathVQA","pathology")

print("SLAKE balance:\n",slake_frame.answer.value_counts())
print("PathVQA balance:\n",pathvqa_frame.answer.value_counts())
print("Cell 5 complete.")


In [ ]:
# CELL 6 — LOAD AND SAFELY PARSE AFRIMED-QA

INVALID={"","n/a","na","none","null","nan"}
def invalid(value):
    if value is None:return True
    try:
        if pd.isna(value):return True
    except Exception:pass
    return str(value).strip().lower() in INVALID

def option_dict(value):
    if isinstance(value,dict):return value
    if not isinstance(value,str):return None
    for parser in (json.loads,ast.literal_eval):
        try:
            parsed=parser(value)
            if isinstance(parsed,dict):return parsed
        except Exception:pass
    return None

def keynorm(value):
    text=str(value).strip().strip("'\"").lower().replace(" ","").replace("_","").replace("-","")
    return text.replace("choice","option")

def keysort(value):
    match=re.search(r"\d+",str(value)); return int(match.group()) if match else 10000

def parse_record(raw_options,raw_answer):
    dictionary=option_dict(raw_options)
    if dictionary is None:return None
    keys=[]; options=[]
    for key,value in sorted(dictionary.items(),key=lambda item:keysort(item[0])):
        if invalid(value):continue
        keys.append(keynorm(key)); options.append(re.sub(r"\s+"," ",str(value).strip()))
    if len(options)<2 or len(set(x.lower() for x in options))<2:return None
    answer=keynorm(raw_answer)
    if answer not in keys:return None
    return options,keys.index(answer),keys

def choose_question(row):
    for column in ["question_clean","question","prompt"]:
        value=row.get(column)
        if not invalid(value) and str(value).strip():return str(value).strip()
    return ""

print("Loading AfriMed-QA...")
source=load_dataset("afrimedqa/afrimedqa_v2",split="train").to_pandas()
candidates=source[source.answer_options.notna() & source.correct_answer.notna()].copy()
if "split" in candidates.columns:
    test=candidates[candidates.split.fillna("").astype(str).str.strip().str.lower().eq("test")]
    if not test.empty:candidates=test
rows=[]
for source_index,row in candidates.iterrows():
    parsed=parse_record(row.get("answer_options"),row.get("correct_answer"))
    if parsed is None:continue
    question=choose_question(row)
    if not question:continue
    options,correct_index,keys=parsed
    country=row.get("country","not specified"); specialty=row.get("specialty","multi-specialty")
    if invalid(country):country="not specified"
    if invalid(specialty):specialty="multi-specialty"
    rows.append({"dataset":"AfriMed-QA","domain":str(specialty),"country":str(country),
                 "african_origin":True,"question":question,"options":options,
                 "option_keys":keys,"correct_index":int(correct_index),"source_index":int(source_index)})
afrimed_frame=pd.DataFrame(rows)
if afrimed_frame.empty:raise ValueError("No valid AfriMed-QA rows")
afrimed_frame=afrimed_frame.sample(min(CFG["n_afrimed"],len(afrimed_frame)),random_state=SEED).reset_index(drop=True)
afrimed_frame["item_id"]=[f"AFR_{i:05d}" for i in range(len(afrimed_frame))]
print("Valid AfriMed pool:",len(rows))
print("Selected AfriMed pilot:",len(afrimed_frame))
print("Cell 6 complete.")


In [ ]:
# CELL 7 — SINGLE EXAMPLE TEST; STOP IF THIS FAILS

test_row=slake_frame.iloc[0]
options=["Yes","No","Image evidence is insufficient"]
prediction,confidence,probabilities=predict_options(test_row.question,options,test_row.image)
print("Question:",test_row.question)
print("Reference:",test_row.answer)
print("Prediction:",options[prediction])
print("Confidence:",confidence)
print("Probabilities:",probabilities)
print("Sum:",sum(probabilities))
assert np.isfinite(confidence) and 0<=confidence<=1
assert abs(sum(probabilities)-1)<1e-5
print("Cell 7 complete. Single-example test passed.")


In [ ]:
# CELL 8 — RUN RESUMABLE 32-EVALUATION PILOT

results=[]; errors=[]
if CFG["resume"] and os.path.exists(PRED_PATH):
    previous=pd.read_csv(PRED_PATH)
    if {"item_id","condition","confidence"}.issubset(previous.columns):
        previous["confidence"]=pd.to_numeric(previous.confidence,errors="coerce")
        previous=previous[np.isfinite(previous.confidence)&previous.confidence.between(0,1)].copy()
        results=previous.to_dict("records")
completed={(str(row["item_id"]),str(row["condition"])) for row in results}

def save():
    pd.DataFrame(results).to_csv(PRED_PATH,index=False)
    if errors:pd.DataFrame(errors).to_csv(ERROR_PATH,index=False)

for frame,prefix in [(slake_frame,"SLK"),(pathvqa_frame,"PTH")]:
    for index,row in tqdm(frame.iterrows(),total=len(frame),desc=frame.iloc[0].dataset):
        item=f"{prefix}_{index:05d}"; opts=["Yes","No","Image evidence is insufficient"]
        truth=0 if row.answer=="yes" else 1
        for condition in CFG["conditions"]:
            key=(item,condition)
            if key in completed:continue
            try:
                pred,conf,probs=predict_options(row.question,opts,perturb_image(row.image,condition))
                results.append({"item_id":item,"dataset":row.dataset,"domain":row.domain,"country":row.country,
                    "condition":condition,"question":row.question,"correct_index":truth,"predicted_index":pred,
                    "predicted_answer":opts[pred],"reference_answer":opts[truth],"confidence":conf,
                    "correct":int(pred==truth),"abstained":int(pred==2),"probabilities":json.dumps(probs)})
                completed.add(key);save()
            except Exception as error:
                errors.append({"item_id":item,"dataset":row.dataset,"condition":condition,"error":repr(error)});save()

for _,row in tqdm(afrimed_frame.iterrows(),total=len(afrimed_frame),desc="AfriMed-QA"):
    key=(row.item_id,"text")
    if key in completed:continue
    try:
        opts=list(row.options); pred,conf,probs=predict_options(row.question,opts,None); truth=int(row.correct_index)
        results.append({"item_id":row.item_id,"dataset":"AfriMed-QA","domain":row.domain,"country":row.country,
            "condition":"text","question":row.question,"correct_index":truth,"predicted_index":pred,
            "predicted_answer":opts[pred],"reference_answer":opts[truth],"confidence":conf,
            "correct":int(pred==truth),"abstained":0,"probabilities":json.dumps(probs)})
        completed.add(key);save()
    except Exception as error:
        errors.append({"item_id":row.item_id,"dataset":"AfriMed-QA","condition":"text","error":repr(error)});save()

results_frame=pd.DataFrame(results).drop_duplicates(["item_id","condition"],keep="last")
results_frame.to_csv(PRED_PATH,index=False)
expected=(len(slake_frame)+len(pathvqa_frame))*len(CFG["conditions"])+len(afrimed_frame)
print("Expected:",expected)
print("Saved valid predictions:",len(results_frame))
print("Errors:",len(errors))
print("Cell 8 complete.")


In [ ]:
# CELL 9 — METRICS, FIGURES, AND OUTPUTS

def binary_auroc(y,scores):
    y=np.asarray(y,int); scores=np.asarray(scores,float)
    positives=(y==1).sum(); negatives=(y==0).sum()
    if positives==0 or negatives==0:return np.nan
    order=np.argsort(scores,kind="mergesort"); sorted_scores=scores[order]; ranks=np.empty(len(scores),float)
    start=0
    while start<len(scores):
        end=start+1
        while end<len(scores) and sorted_scores[end]==sorted_scores[start]:end+=1
        ranks[order[start:end]]=(start+1+end)/2; start=end
    return float((ranks[y==1].sum()-positives*(positives+1)/2)/(positives*negatives))

def ece(y,c,bins):
    y=np.asarray(y,float); c=np.asarray(c,float); total=0
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        mask=(c>=lo)&(c<=hi if hi==1 else c<hi)
        if mask.any():total+=mask.mean()*abs(y[mask].mean()-c[mask].mean())
    return float(total)

def summarize(group):
    y=group.correct.to_numpy(int); c=group.confidence.to_numpy(float)
    return pd.Series({"n":len(group),"accuracy":y.mean(),"mean_confidence":c.mean(),
        "ece":ece(y,c,CFG["ece_bins"]),"brier":np.mean((c-y)**2),
        "confidence_auroc":binary_auroc(y,c),"abstention_rate":group.abstained.mean(),
        "high_confidence_error_rate":((y==0)&(c>=CFG["high_confidence"])).mean()})

results_frame=pd.read_csv(PRED_PATH)
results_frame["confidence"]=pd.to_numeric(results_frame.confidence,errors="coerce")
results_frame=results_frame[np.isfinite(results_frame.confidence)&results_frame.confidence.between(0,1)].copy()
try:
    summary=results_frame.groupby(["dataset","condition"],group_keys=False).apply(summarize,include_groups=False).reset_index()
except TypeError:
    summary=results_frame.groupby(["dataset","condition"],group_keys=False).apply(summarize).reset_index()
summary.to_csv(os.path.join(OUT,"summary.csv"),index=False)
print(summary.round(4).to_string(index=False))

fig,axes=plt.subplots(1,3,figsize=(16,4.5))
sns.barplot(data=summary,x="dataset",y="accuracy",hue="condition",ax=axes[0]);axes[0].set_ylim(0,1);axes[0].set_title("Accuracy")
sns.barplot(data=summary,x="dataset",y="ece",hue="condition",ax=axes[1]);axes[1].set_title("Expected calibration error")
sns.barplot(data=summary,x="dataset",y="high_confidence_error_rate",hue="condition",ax=axes[2]);axes[2].set_ylim(0,1);axes[2].set_title("High-confidence errors")
for axis in axes:axis.tick_params(axis="x",rotation=20)
plt.tight_layout();plt.savefig(os.path.join(OUT,"audit_summary.png"),dpi=180,bbox_inches="tight");plt.show()

metadata={"model":CFG["model"],"sample_sizes":{"SLAKE":len(slake_frame),"PathVQA":len(pathvqa_frame),"AfriMed-QA":len(afrimed_frame)},
"conditions":CFG["conditions"],"research_only":True,"confidence_definition":"Softmax over next-token logits restricted to supplied option letters."}
with open(os.path.join(OUT,"metadata.json"),"w") as file:json.dump(metadata,file,indent=2)
print("Outputs:",sorted(os.listdir(OUT)))
print("Cell 9 complete. Clean pilot finished.")


In [ ]:
import os

working_directory = "/kaggle/working"

print("Folders and files in Kaggle working directory:")

for item in sorted(os.listdir(working_directory)):
    full_path = os.path.join(
        working_directory,
        item,
    )

    if os.path.isdir(full_path):
        print("FOLDER:", full_path)
    else:
        print("FILE:", full_path)

In [ ]:
import glob
import os
import pandas as pd

files = glob.glob(
    "/kaggle/working/**/predictions.csv",
    recursive=True
)

print("Files found:", len(files))

for path in files:
    frame = pd.read_csv(path)
    print(path)
    print("Rows:", len(frame))

print("Search complete.")

In [ ]:
model_ready = "model" in globals()

slake_ready = "slake_frame" in globals()
pathvqa_ready = "pathvqa_frame" in globals()
afrimed_ready = "afrimed_frame" in globals()

predict_ready = "predict_options" in globals()
perturb_ready = "perturb_image" in globals()

print("Model ready:", model_ready)
print("SLAKE ready:", slake_ready)
print("PathVQA ready:", pathvqa_ready)
print("AfriMed-QA ready:", afrimed_ready)
print("Prediction function ready:", predict_ready)
print("Image function ready:", perturb_ready)

In [ ]:
if "visual_conditions" not in CFG:
    CFG["visual_conditions"] = CFG.get(
        "conditions",
        ["clean", "blur", "dark"],
    )

if "conditions" not in CFG:
    CFG["conditions"] = CFG["visual_conditions"]

print(
    "Visual conditions:",
    CFG["visual_conditions"],
)

print("Configuration corrected successfully.")

In [ ]:
required_objects = [
    "CFG",
    "model",
    "processor",
    "slake_frame",
    "pathvqa_frame",
    "afrimed_frame",
    "predict_options",
    "perturb_image",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing objects: "
        + ", ".join(missing_objects)
    )

print("=" * 60)
print("READY FOR EVALUATION")
print("=" * 60)

print("SLAKE cases:", len(slake_frame))
print("PathVQA cases:", len(pathvqa_frame))
print("AfriMed-QA cases:", len(afrimed_frame))
print("Model device:", next(model.parameters()).device)
print("Evaluation mode:", not model.training)

expected_total = (
    (
        len(slake_frame)
        + len(pathvqa_frame)
    )
    * len(CFG["visual_conditions"])
    + len(afrimed_frame)
)

print("Expected total evaluations:", expected_total)
print("\nAll components are ready.")

In [ ]:
# CELL 8: RUN THE COMPLETE 32-EVALUATION PILOT

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Check required objects
# ------------------------------------------------------------

required_objects = [
    "CFG",
    "model",
    "processor",
    "slake_frame",
    "pathvqa_frame",
    "afrimed_frame",
    "predict_options",
    "perturb_image",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# Correct configuration names
# ------------------------------------------------------------

if "visual_conditions" not in CFG:
    CFG["visual_conditions"] = CFG.get(
        "conditions",
        ["clean", "blur", "dark"],
    )

if "conditions" not in CFG:
    CFG["conditions"] = CFG["visual_conditions"]

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUT = CFG.get(
    "output_dir",
    "/kaggle/working/healthcare_audit_pilot",
)

os.makedirs(OUT, exist_ok=True)

PRED_PATH = os.path.join(
    OUT,
    "predictions.csv",
)

ERROR_PATH = os.path.join(
    OUT,
    "evaluation_errors.csv",
)

# ------------------------------------------------------------
# Resume previous progress
# ------------------------------------------------------------

results = []
errors = []

if os.path.isfile(PRED_PATH):
    previous = pd.read_csv(PRED_PATH)

    required_columns = {
        "item_id",
        "condition",
        "confidence",
    }

    if required_columns.issubset(previous.columns):
        previous["confidence"] = pd.to_numeric(
            previous["confidence"],
            errors="coerce",
        )

        valid_previous = (
            np.isfinite(previous["confidence"])
            & previous["confidence"].between(0.0, 1.0)
        )

        previous = previous[
            valid_previous
        ].copy()

        results = previous.to_dict("records")

if os.path.isfile(ERROR_PATH):
    try:
        errors = pd.read_csv(
            ERROR_PATH
        ).to_dict("records")
    except Exception:
        errors = []

completed = {
    (
        str(record["item_id"]),
        str(record["condition"]),
    )
    for record in results
}

# ------------------------------------------------------------
# Save helper
# ------------------------------------------------------------

def save_progress():
    pd.DataFrame(results).to_csv(
        PRED_PATH,
        index=False,
    )

    if errors:
        pd.DataFrame(errors).to_csv(
            ERROR_PATH,
            index=False,
        )

# ------------------------------------------------------------
# Expected evaluation count
# ------------------------------------------------------------

expected_visual = (
    len(slake_frame) + len(pathvqa_frame)
) * len(CFG["visual_conditions"])

expected_total = (
    expected_visual + len(afrimed_frame)
)

print("=" * 60)
print("RUNNING 32-EVALUATION PILOT")
print("=" * 60)
print("SLAKE cases:", len(slake_frame))
print("PathVQA cases:", len(pathvqa_frame))
print("AfriMed-QA cases:", len(afrimed_frame))
print("Existing predictions:", len(results))
print("Expected total evaluations:", expected_total)
print("Output directory:", OUT)

# ------------------------------------------------------------
# Evaluate SLAKE and PathVQA
# ------------------------------------------------------------

visual_datasets = [
    (slake_frame, "SLK"),
    (pathvqa_frame, "PTH"),
]

for frame, prefix in visual_datasets:
    dataset_name = str(
        frame.iloc[0]["dataset"]
    )

    print("\nEvaluating:", dataset_name)

    for row_index, row in tqdm(
        frame.iterrows(),
        total=len(frame),
        desc=dataset_name,
    ):
        item_id = (
            f"{prefix}_{row_index:05d}"
        )

        options = [
            "Yes",
            "No",
            "Image evidence is insufficient",
        ]

        correct_index = (
            0 if row["answer"] == "yes" else 1
        )

        for condition in CFG["visual_conditions"]:
            key = (
                item_id,
                condition,
            )

            if key in completed:
                continue

            try:
                image = perturb_image(
                    row["image"],
                    condition,
                )

                predicted_index, confidence, probabilities = (
                    predict_options(
                        question=row["question"],
                        options=options,
                        image=image,
                    )
                )

                results.append({
                    "item_id": item_id,
                    "dataset": dataset_name,
                    "domain": str(row["domain"]),
                    "country": str(row["country"]),
                    "african_origin": False,
                    "condition": condition,
                    "question": str(row["question"]),
                    "correct_index": int(correct_index),
                    "predicted_index": int(predicted_index),
                    "predicted_answer": options[predicted_index],
                    "reference_answer": options[correct_index],
                    "confidence": float(confidence),
                    "correct": int(
                        predicted_index == correct_index
                    ),
                    "abstained": int(
                        predicted_index == 2
                    ),
                    "probabilities": json.dumps(
                        probabilities
                    ),
                })

                completed.add(key)

            except Exception as error:
                errors.append({
                    "item_id": item_id,
                    "dataset": dataset_name,
                    "condition": condition,
                    "question": str(row["question"]),
                    "error": repr(error),
                })

            save_progress()

# ------------------------------------------------------------
# Evaluate AfriMed-QA
# ------------------------------------------------------------

print("\nEvaluating: AfriMed-QA")

for _, row in tqdm(
    afrimed_frame.iterrows(),
    total=len(afrimed_frame),
    desc="AfriMed-QA",
):
    item_id = str(row["item_id"])
    condition = "text"

    key = (
        item_id,
        condition,
    )

    if key in completed:
        continue

    try:
        options = list(row["options"])

        predicted_index, confidence, probabilities = (
            predict_options(
                question=row["question"],
                options=options,
                image=None,
            )
        )

        correct_index = int(
            row["correct_index"]
        )

        results.append({
            "item_id": item_id,
            "dataset": "AfriMed-QA",
            "domain": str(row["domain"]),
            "country": str(row["country"]),
            "african_origin": True,
            "condition": condition,
            "question": str(row["question"]),
            "correct_index": correct_index,
            "predicted_index": int(predicted_index),
            "predicted_answer": options[predicted_index],
            "reference_answer": options[correct_index],
            "confidence": float(confidence),
            "correct": int(
                predicted_index == correct_index
            ),
            "abstained": 0,
            "probabilities": json.dumps(
                probabilities
            ),
        })

        completed.add(key)

    except Exception as error:
        errors.append({
            "item_id": item_id,
            "dataset": "AfriMed-QA",
            "condition": condition,
            "question": str(row["question"]),
            "error": repr(error),
        })

    save_progress()

# ------------------------------------------------------------
# Validate and save results
# ------------------------------------------------------------

results_frame = pd.DataFrame(results)

if results_frame.empty:
    raise ValueError(
        "No predictions were generated. "
        "Check evaluation_errors.csv."
    )

results_frame["confidence"] = pd.to_numeric(
    results_frame["confidence"],
    errors="coerce",
)

valid_rows = (
    np.isfinite(results_frame["confidence"])
    & results_frame["confidence"].between(
        0.0,
        1.0,
    )
)

invalid_rows = results_frame[
    ~valid_rows
].copy()

if not invalid_rows.empty:
    invalid_rows.to_csv(
        os.path.join(
            OUT,
            "invalid_predictions.csv",
        ),
        index=False,
    )

results_frame = results_frame[
    valid_rows
].copy()

results_frame = (
    results_frame
    .drop_duplicates(
        subset=[
            "item_id",
            "condition",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

results_frame.to_csv(
    PRED_PATH,
    index=False,
)

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PILOT EVALUATION COMPLETE")
print("=" * 60)
print("Expected evaluations:", expected_total)
print("Saved valid predictions:", len(results_frame))
print("Errors recorded:", len(errors))
print("Prediction file:", PRED_PATH)

print("\nPredictions by dataset:")
print(
    results_frame["dataset"].value_counts()
)

print("\nPredictions by condition:")
print(
    results_frame["condition"].value_counts()
)

if len(results_frame) == expected_total:
    print(
        "\nAll 32 evaluations completed successfully."
    )
else:
    print(
        "\nSome evaluations are missing."
    )
    print(
        "Run Cell 8 again to retry missing items."
    )

print("\nCell 8 completed successfully.")

In [ ]:
# =============================================================================
# CELL 9: SAFE AFRIMED-QA PARSER
# =============================================================================

import ast
import json
import re

import numpy as np
import pandas as pd


INVALID_OPTION_VALUES = {
    "",
    "n/a",
    "na",
    "none",
    "null",
    "nan",
}


def is_missing_value(value):
    """Check whether a scalar value is empty or missing."""

    if value is None:
        return True

    if isinstance(
        value,
        (list, tuple, dict, np.ndarray),
    ):
        return False

    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    return (
        str(value).strip().lower()
        in INVALID_OPTION_VALUES
    )


def option_sort_key(key):
    """Sort option1, option2, option3, etc."""

    text = str(key).strip()

    number_match = re.search(
        r"\d+",
        text,
    )

    if number_match:
        return (
            0,
            int(number_match.group()),
        )

    letter_match = re.search(
        r"[A-Ha-h]",
        text,
    )

    if letter_match:
        return (
            1,
            ord(letter_match.group().upper())
            - ord("A"),
        )

    return (2, text.lower())


def normalize_option_key(value):
    """
    Normalize option labels such as:
    option4, Option 4, option_4, choice4, A, or optionA.
    """

    if value is None:
        return ""

    text = (
        str(value)
        .strip()
        .strip("'\"")
        .lower()
    )

    text = re.sub(
        r"^(?:correct\s+answer|answer)"
        r"\s*(?:is|:|=|-)?\s*",
        "",
        text,
    )

    text = (
        text
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
        .replace(")", "")
        .replace("(", "")
        .replace(":", "")
    )

    text = text.replace(
        "choice",
        "option",
    )

    # Convert A, B, C, etc. to option1, option2, option3.
    if re.fullmatch(r"[a-h]", text):
        number = (
            ord(text.upper())
            - ord("A")
            + 1
        )

        return f"option{number}"

    # Convert optionA to option1.
    option_letter_match = re.fullmatch(
        r"option([a-h])",
        text,
    )

    if option_letter_match:
        number = (
            ord(
                option_letter_match
                .group(1)
                .upper()
            )
            - ord("A")
            + 1
        )

        return f"option{number}"

    # Convert a bare number to option<number>.
    if text.isdigit():
        return f"option{text}"

    return text


def normalize_option_text(value):
    """Normalize option text for duplicate checking."""

    if is_missing_value(value):
        return ""

    text = str(value).strip().lower()

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def parse_answer_options(answer_options):
    """
    Convert serialized answer options into a dictionary.

    Supports native dictionaries, JSON strings,
    and Python dictionary strings.
    """

    if isinstance(answer_options, dict):
        return answer_options

    if not isinstance(answer_options, str):
        return None

    text = answer_options.strip()

    if not text:
        return None

    try:
        parsed = json.loads(text)

        if isinstance(parsed, dict):
            return parsed

    except (
        json.JSONDecodeError,
        TypeError,
    ):
        pass

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, dict):
            return parsed

    except (
        ValueError,
        SyntaxError,
        TypeError,
    ):
        pass

    return None


def parse_afrimed_record(
    answer_options,
    correct_answer,
):
    """
    Parse one AfriMed-QA multiple-choice record.

    Invalid options such as n/a are removed while
    the original option keys are preserved.

    Returns:
        options
        correct_index
        option_keys
    """

    option_dictionary = parse_answer_options(
        answer_options
    )

    if option_dictionary is None:
        return None, None, None

    retained_keys = []
    retained_options = []

    sorted_items = sorted(
        option_dictionary.items(),
        key=lambda item: option_sort_key(
            item[0]
        ),
    )

    for key, value in sorted_items:

        if is_missing_value(value):
            continue

        cleaned_value = re.sub(
            r"\s+",
            " ",
            str(value).strip(),
        )

        normalized_key = normalize_option_key(
            key
        )

        if not normalized_key:
            continue

        if not cleaned_value:
            continue

        retained_keys.append(
            normalized_key
        )

        retained_options.append(
            cleaned_value
        )

    # Require at least two meaningful options.
    if len(retained_options) < 2:
        return None, None, None

    normalized_texts = [
        normalize_option_text(option)
        for option in retained_options
    ]

    # Reject duplicate answer options.
    if len(set(normalized_texts)) < 2:
        return None, None, None

    normalized_answer = normalize_option_key(
        correct_answer
    )

    # Exclude rows where the answer points to
    # an invalid or removed option.
    if normalized_answer not in retained_keys:
        return None, None, None

    correct_index = retained_keys.index(
        normalized_answer
    )

    return (
        retained_options,
        correct_index,
        retained_keys,
    )


def choose_afrimed_question(row):
    """Choose the best available question field."""

    for column in [
        "question_clean",
        "question",
        "prompt",
    ]:
        value = row.get(column)

        if not is_missing_value(value):
            question = str(value).strip()

            if question:
                return question

    return ""


def build_afrimed_evaluation_frame(
    source_frame,
    maximum_items=8,
    seed=42,
):
    """Create a valid and reproducible AfriMed-QA sample."""

    valid_rows = []

    option_failures = 0
    answer_failures = 0
    question_failures = 0

    for source_index, row in source_frame.iterrows():

        raw_options = row.get(
            "answer_options"
        )

        raw_answer = row.get(
            "correct_answer"
        )

        (
            options,
            correct_index,
            option_keys,
        ) = parse_afrimed_record(
            answer_options=raw_options,
            correct_answer=raw_answer,
        )

        if options is None:

            parsed_dictionary = parse_answer_options(
                raw_options
            )

            if parsed_dictionary is None:
                option_failures += 1

            else:
                available_keys = [
                    normalize_option_key(key)
                    for key, value
                    in parsed_dictionary.items()
                    if not is_missing_value(value)
                ]

                normalized_answer = normalize_option_key(
                    raw_answer
                )

                if normalized_answer not in available_keys:
                    answer_failures += 1
                else:
                    option_failures += 1

            continue

        question = choose_afrimed_question(
            row
        )

        if not question:
            question_failures += 1
            continue

        specialty = row.get(
            "specialty",
            "multi-specialty",
        )

        country = row.get(
            "country",
            "not specified",
        )

        if is_missing_value(specialty):
            specialty = "multi-specialty"

        if is_missing_value(country):
            country = "not specified"

        valid_rows.append({
            "dataset": "AfriMed-QA",
            "domain": str(specialty).strip(),
            "country": str(country).strip(),
            "african_origin": True,
            "question": question,
            "options": options,
            "option_keys": option_keys,
            "correct_index": int(
                correct_index
            ),
            "source_index": int(
                source_index
            ),
        })

    evaluation_frame = pd.DataFrame(
        valid_rows
    )

    print("=" * 60)
    print("AFRIMED-QA PARSING REPORT")
    print("=" * 60)

    print(
        "Rows examined:",
        len(source_frame),
    )

    print(
        "Valid rows:",
        len(evaluation_frame),
    )

    print(
        "Option parsing failures:",
        option_failures,
    )

    print(
        "Answer alignment failures:",
        answer_failures,
    )

    print(
        "Missing-question failures:",
        question_failures,
    )

    if evaluation_frame.empty:
        raise ValueError(
            "No valid AfriMed-QA records were produced."
        )

    sample_size = min(
        maximum_items,
        len(evaluation_frame),
    )

    evaluation_frame = (
        evaluation_frame
        .sample(
            n=sample_size,
            random_state=seed,
        )
        .reset_index(drop=True)
    )

    evaluation_frame["item_id"] = [
        f"AFR_{index:05d}"
        for index in range(
            len(evaluation_frame)
        )
    ]

    print(
        "Selected evaluation rows:",
        len(evaluation_frame),
    )

    return evaluation_frame


# Test the parser before using the real dataset.
test_answer_options = {
    "option1": "Laparoscopic adjustable gastric banding",
    "option2": "Roux-en-Y gastric bypass",
    "option3": "Laparoscopic sleeve gastrectomy",
    "option4": "Biliopancreatic diversion with duodenal switch",
    "option5": "n/a",
}

(
    test_options,
    test_correct_index,
    test_option_keys,
) = parse_afrimed_record(
    answer_options=test_answer_options,
    correct_answer="option4",
)

assert test_option_keys == [
    "option1",
    "option2",
    "option3",
    "option4",
]

assert test_correct_index == 3

assert (
    test_options[test_correct_index]
    == "Biliopancreatic diversion with duodenal switch"
)

print("\nParser test passed.")

print(
    "Cell 9 complete: "
    "AfriMed-QA parser is ready."
)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = "/kaggle/working/healthcare_audit_clean"

PRED_PATH = os.path.join(
    OUT,
    "predictions.csv",
)

if not os.path.isfile(PRED_PATH):
    raise FileNotFoundError(
        "predictions.csv was not found at: "
        + PRED_PATH
    )

results = pd.read_csv(PRED_PATH)

print("Predictions loaded:", len(results))

if len(results) != 32:
    raise ValueError(
        "Expected 32 predictions, but found "
        + str(len(results))
    )

results["confidence"] = pd.to_numeric(
    results["confidence"],
    errors="coerce",
)

results["correct"] = pd.to_numeric(
    results["correct"],
    errors="coerce",
)

valid_rows = (
    np.isfinite(results["confidence"])
    & np.isfinite(results["correct"])
    & results["confidence"].between(
        0.0,
        1.0,
    )
)

results = results[
    valid_rows
].copy()

print(
    "Valid numerical predictions:",
    len(results),
)


def calculate_ece(
    correctness,
    confidence,
    number_of_bins=8,
):
    correctness = np.asarray(
        correctness,
        dtype=float,
    )

    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    total_error = 0.0

    for lower, upper in zip(
        edges[:-1],
        edges[1:],
    ):
        if upper == 1.0:
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if mask.any():
            total_error += (
                mask.mean()
                * abs(
                    correctness[mask].mean()
                    - confidence[mask].mean()
                )
            )

    return float(total_error)


def calculate_auroc(
    correctness,
    confidence,
):
    correctness = np.asarray(
        correctness,
        dtype=int,
    )

    confidence = np.asarray(
        confidence,
        dtype=float,
    )

    positive_count = int(
        (correctness == 1).sum()
    )

    negative_count = int(
        (correctness == 0).sum()
    )

    if (
        positive_count == 0
        or negative_count == 0
    ):
        return np.nan

    order = np.argsort(
        confidence,
        kind="mergesort",
    )

    sorted_scores = confidence[
        order
    ]

    ranks = np.empty(
        len(confidence),
        dtype=float,
    )

    start = 0

    while start < len(confidence):
        end = start + 1

        while (
            end < len(confidence)
            and sorted_scores[end]
            == sorted_scores[start]
        ):
            end += 1

        ranks[
            order[start:end]
        ] = (
            start + 1 + end
        ) / 2.0

        start = end

    positive_rank_sum = ranks[
        correctness == 1
    ].sum()

    auc = (
        positive_rank_sum
        - positive_count
        * (positive_count + 1)
        / 2.0
    ) / (
        positive_count
        * negative_count
    )

    return float(auc)


summary_rows = []

grouped_results = results.groupby(
    [
        "dataset",
        "condition",
    ]
)

for group_key, group in grouped_results:
    dataset_name = group_key[0]
    condition_name = group_key[1]

    correctness = group[
        "correct"
    ].to_numpy(dtype=int)

    confidence = group[
        "confidence"
    ].to_numpy(dtype=float)

    high_confidence_errors = (
        (correctness == 0)
        & (confidence >= 0.80)
    )

    summary_rows.append({
        "dataset": dataset_name,
        "condition": condition_name,
        "n": len(group),
        "accuracy": float(
            correctness.mean()
        ),
        "mean_confidence": float(
            confidence.mean()
        ),
        "ece": calculate_ece(
            correctness,
            confidence,
            number_of_bins=8,
        ),
        "brier": float(
            np.mean(
                (
                    confidence
                    - correctness
                ) ** 2
            )
        ),
        "confidence_auroc": (
            calculate_auroc(
                correctness,
                confidence,
            )
        ),
        "abstention_rate": float(
            group["abstained"].mean()
        ),
        "high_confidence_error_rate": float(
            high_confidence_errors.mean()
        ),
    })


summary = pd.DataFrame(
    summary_rows
)

SUMMARY_PATH = os.path.join(
    OUT,
    "summary.csv",
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print("\n" + "=" * 60)
print("FINAL PILOT RESULTS")
print("=" * 60)

print(
    summary
    .round(4)
    .to_string(index=False)
)


figure, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4.5),
)

for dataset_name, group in summary.groupby(
    "dataset"
):
    conditions = group[
        "condition"
    ].tolist()

    axes[0].plot(
        conditions,
        group["accuracy"],
        marker="o",
        label=dataset_name,
    )

    axes[1].plot(
        conditions,
        group["ece"],
        marker="o",
        label=dataset_name,
    )

    axes[2].plot(
        conditions,
        group[
            "high_confidence_error_rate"
        ],
        marker="o",
        label=dataset_name,
    )

axes[0].set_title("Accuracy")
axes[0].set_ylim(0.0, 1.0)

axes[1].set_title(
    "Expected Calibration Error"
)

axes[2].set_title(
    "High-Confidence Error Rate"
)
axes[2].set_ylim(0.0, 1.0)

for axis in axes:
    axis.set_xlabel("Condition")
    axis.grid(alpha=0.2)
    axis.legend()

plt.tight_layout()

FIGURE_PATH = os.path.join(
    OUT,
    "audit_summary.png",
)

plt.savefig(
    FIGURE_PATH,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


metadata = {
    "model": "llava-hf/llava-1.5-7b-hf",
    "total_evaluations": len(results),
    "sample_sizes": {
        "SLAKE": 4,
        "PathVQA": 4,
        "AfriMed-QA": 8,
    },
    "visual_conditions": [
        "clean",
        "blur",
        "dark",
    ],
    "research_only": True,
    "confidence_definition": (
        "Softmax over next-token logits "
        "restricted to supplied option letters."
    ),
}

METADATA_PATH = os.path.join(
    OUT,
    "metadata.json",
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        metadata,
        metadata_file,
        indent=2,
    )


print("\n" + "=" * 60)
print("OUTPUTS SAVED")
print("=" * 60)

for file_name in sorted(
    os.listdir(OUT)
):
    print(file_name)

print("\nOutput directory:", OUT)

print(
    "Final metrics completed successfully."
)

In [ ]:
import os
import pandas as pd

output_directory = (
    "/kaggle/working/healthcare_audit_clean"
)

prediction_path = os.path.join(
    output_directory,
    "predictions.csv",
)

predictions = pd.read_csv(
    prediction_path
)

incorrect = predictions[
    predictions["correct"] == 0
].copy()

incorrect = incorrect.sort_values(
    "confidence",
    ascending=False,
)

columns = [
    "dataset",
    "condition",
    "question",
    "reference_answer",
    "predicted_answer",
    "confidence",
    "abstained",
]

columns = [
    column
    for column in columns
    if column in incorrect.columns
]

print(
    "Total incorrect answers:",
    len(incorrect),
)

display(
    incorrect[columns]
)

high_confidence_errors = incorrect[
    incorrect["confidence"] >= 0.80
].copy()

print(
    "\nHigh-confidence errors:",
    len(high_confidence_errors),
)

display(
    high_confidence_errors[columns]
)

high_confidence_errors.to_csv(
    os.path.join(
        output_directory,
        "high_confidence_errors.csv",
    ),
    index=False,
)

In [ ]:
import os
import shutil

current_folder = (
    "/kaggle/working/healthcare_audit_clean"
)

pilot_folder = (
    "/kaggle/working/healthcare_audit_pilot_32"
)

os.makedirs(
    pilot_folder,
    exist_ok=True,
)

for file_name in os.listdir(current_folder):
    source = os.path.join(
        current_folder,
        file_name,
    )

    destination = os.path.join(
        pilot_folder,
        file_name,
    )

    if os.path.isfile(source):
        shutil.copy2(
            source,
            destination,
        )

print("Pilot preserved in:", pilot_folder)
print("Files:", sorted(os.listdir(pilot_folder)))

In [ ]:
CFG["n_slake"] = 20
CFG["n_pathvqa"] = 20
CFG["n_afrimed"] = 40

CFG["output_directory"] = (
    "/kaggle/working/healthcare_audit_baseline_160"
)

os.makedirs(
    CFG["output_directory"],
    exist_ok=True,
)

PREDICTION_PATH = os.path.join(
    CFG["output_directory"],
    "predictions.csv",
)

ERROR_PATH = os.path.join(
    CFG["output_directory"],
    "evaluation_errors.csv",
)

print("Baseline output folder:", CFG["output_directory"])

In [ ]:
results = []
evaluation_errors = []
completed = set()

print("Checkpoint state reset for the larger baseline.")

In [ ]:
import os
import pandas as pd

pilot_dir = "/kaggle/working/healthcare_audit_clean"
pilot_predictions = pd.read_csv(
    os.path.join(pilot_dir, "predictions.csv")
)

incorrect = pilot_predictions[
    pilot_predictions["correct"] == 0
].copy()

incorrect = incorrect.sort_values(
    "confidence",
    ascending=False
)

columns = [
    "dataset",
    "condition",
    "question",
    "reference_answer",
    "predicted_answer",
    "confidence",
    "abstained",
]

columns = [
    column
    for column in columns
    if column in incorrect.columns
]

print("Incorrect predictions:", len(incorrect))
display(incorrect[columns])

high_confidence_errors = incorrect[
    incorrect["confidence"] >= 0.80
].copy()

print(
    "\nHigh-confidence errors:",
    len(high_confidence_errors)
)

display(high_confidence_errors[columns])

high_confidence_errors.to_csv(
    os.path.join(
        pilot_dir,
        "high_confidence_errors.csv"
    ),
    index=False
)

In [ ]:
import os
import shutil

source_dir = "/kaggle/working/healthcare_audit_clean"
pilot_dir = "/kaggle/working/healthcare_audit_pilot_32"

os.makedirs(pilot_dir, exist_ok=True)

for file_name in os.listdir(source_dir):
    source_path = os.path.join(source_dir, file_name)
    destination_path = os.path.join(pilot_dir, file_name)

    if os.path.isfile(source_path):
        shutil.copy2(source_path, destination_path)

print("Pilot preserved in:", pilot_dir)
print("Files:", sorted(os.listdir(pilot_dir)))

In [ ]:
print("SLAKE:", len(slake_frame))
print("PathVQA:", len(pathvqa_frame))
print("AfriMed-QA:", len(afrimed_frame))

assert len(slake_frame) == 20
assert len(pathvqa_frame) == 20
assert len(afrimed_frame) == 40

print("Baseline datasets are ready.")

In [ ]:
required_objects = [
    "model",
    "processor",
    "slake_frame",
    "pathvqa_frame",
    "afrimed_frame",
    "predict_options",
    "perturb_image",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

print("Missing objects:", missing_objects)

if not missing_objects:
    print("Everything required for evaluation is ready.")
    print("Model device:", next(model.parameters()).device)
    print("SLAKE:", len(slake_frame))
    print("PathVQA:", len(pathvqa_frame))
    print("AfriMed-QA:", len(afrimed_frame))

In [ ]:
required_objects = [
    "model",
    "processor",
    "slake_frame",
    "pathvqa_frame",
    "afrimed_frame",
    "predict_options",
    "perturb_image",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

print("Missing objects:", missing_objects)

if not missing_objects:
    print("Everything required for evaluation is ready.")
    print("Model device:", next(model.parameters()).device)
    print("SLAKE:", len(slake_frame))
    print("PathVQA:", len(pathvqa_frame))
    print("AfriMed-QA:", len(afrimed_frame))

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

CFG["output_directory"] = (
    "/kaggle/working/healthcare_audit_baseline_160"
)
CFG["checkpoint_every"] = 10

os.makedirs(CFG["output_directory"], exist_ok=True)

PREDICTION_PATH = os.path.join(
    CFG["output_directory"],
    "predictions.csv",
)

ERROR_PATH = os.path.join(
    CFG["output_directory"],
    "evaluation_errors.csv",
)

results = []
evaluation_errors = []

if os.path.exists(PREDICTION_PATH):
    old = pd.read_csv(PREDICTION_PATH)

    if {
        "item_id",
        "condition",
        "confidence",
    }.issubset(old.columns):

        old["confidence"] = pd.to_numeric(
            old["confidence"],
            errors="coerce",
        )

        old = old[
            np.isfinite(old["confidence"])
            & old["confidence"].between(0, 1)
        ].copy()

        results = old.to_dict("records")

completed = {
    (
        str(row["item_id"]),
        str(row["condition"]),
    )
    for row in results
}

def save_progress():
    pd.DataFrame(results).to_csv(
        PREDICTION_PATH,
        index=False,
    )

    if evaluation_errors:
        pd.DataFrame(evaluation_errors).to_csv(
            ERROR_PATH,
            index=False,
        )

print("Output:", CFG["output_directory"])
print("Existing valid predictions:", len(results))
print("Expected total: 160")
print("Cell A complete.")

In [ ]:
new_count = 0

visual_sets = [
    (slake_frame, "SLK"),
    (pathvqa_frame, "PTH"),
]

for frame, prefix in visual_sets:

    dataset_name = str(frame.iloc[0]["dataset"])

    for row_index, row in tqdm(
        frame.iterrows(),
        total=len(frame),
        desc=dataset_name,
    ):

        item_id = f"{prefix}_{row_index:05d}"

        options = [
            "Yes",
            "No",
            "Image evidence is insufficient",
        ]

        correct_index = (
            0 if row["answer"] == "yes" else 1
        )

        for condition in CFG["visual_conditions"]:

            key = (item_id, condition)

            if key in completed:
                continue

            try:
                image = perturb_image(
                    row["image"],
                    condition,
                )

                pred, confidence, probabilities = (
                    predict_options(
                        question=row["question"],
                        options=options,
                        image=image,
                    )
                )

                results.append({
                    "item_id": item_id,
                    "dataset": dataset_name,
                    "domain": str(row["domain"]),
                    "country": str(row["country"]),
                    "condition": condition,
                    "question": str(row["question"]),
                    "correct_index": int(correct_index),
                    "predicted_index": int(pred),
                    "predicted_answer": options[pred],
                    "reference_answer": options[correct_index],
                    "confidence": float(confidence),
                    "correct": int(pred == correct_index),
                    "abstained": int(pred == 2),
                    "probabilities": json.dumps(probabilities),
                })

                completed.add(key)
                new_count += 1

            except Exception as error:
                evaluation_errors.append({
                    "item_id": item_id,
                    "dataset": dataset_name,
                    "condition": condition,
                    "error": repr(error),
                })

            if new_count % CFG["checkpoint_every"] == 0:
                save_progress()

save_progress()

print("Visual evaluation finished.")
print("Current prediction rows:", len(results))
print("Errors:", len(evaluation_errors))


for _, row in tqdm(
    afrimed_frame.iterrows(),
    total=len(afrimed_frame),
    desc="AfriMed-QA",
):

    item_id = str(row["item_id"])
    key = (item_id, "text")

    if key in completed:
        continue

    try:
        options = list(row["options"])

        pred, confidence, probabilities = (
            predict_options(
                question=row["question"],
                options=options,
                image=None,
            )
        )

        correct_index = int(
            row["correct_index"]
        )

        results.append({
            "item_id": item_id,
            "dataset": "AfriMed-QA",
            "domain": str(row["domain"]),
            "country": str(row["country"]),
            "condition": "text",
            "question": str(row["question"]),
            "correct_index": correct_index,
            "predicted_index": int(pred),
            "predicted_answer": options[pred],
            "reference_answer": options[correct_index],
            "confidence": float(confidence),
            "correct": int(pred == correct_index),
            "abstained": 0,
            "probabilities": json.dumps(probabilities),
        })

        completed.add(key)
        new_count += 1

    except Exception as error:
        evaluation_errors.append({
            "item_id": item_id,
            "dataset": "AfriMed-QA",
            "condition": "text",
            "error": repr(error),
        })

    if new_count % CFG["checkpoint_every"] == 0:
        save_progress()

save_progress()

final_predictions = pd.DataFrame(results)

final_predictions["confidence"] = pd.to_numeric(
    final_predictions["confidence"],
    errors="coerce",
)

final_predictions = final_predictions[
    np.isfinite(final_predictions["confidence"])
    & final_predictions["confidence"].between(0, 1)
].copy()

final_predictions = final_predictions.drop_duplicates(
    subset=["item_id", "condition"],
    keep="last",
)

final_predictions.to_csv(
    PREDICTION_PATH,
    index=False,
)

print("\n" + "=" * 60)
print("BASELINE EVALUATION FINISHED")
print("=" * 60)
print("Valid predictions:", len(final_predictions))
print("Evaluation errors:", len(evaluation_errors))

print("\nRows by dataset:")
print(final_predictions["dataset"].value_counts())

print("\nRows by condition:")
print(final_predictions["condition"].value_counts())

if len(final_predictions) == 160:
    print("\nSUCCESS: All 160 evaluations completed.")
else:
    print(
        "\nMissing predictions:",
        160 - len(final_predictions),
    )
    print("Rerun this cell to resume missing cases.")

In [ ]:
import os
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"

prediction_path = os.path.join(
    OUT,
    "predictions.csv",
)

predictions = pd.read_csv(
    prediction_path
)

incorrect_predictions = predictions[
    predictions["correct"] == 0
].copy()

incorrect_predictions = incorrect_predictions.sort_values(
    "confidence",
    ascending=False,
)

high_confidence_errors = incorrect_predictions[
    incorrect_predictions["confidence"] >= 0.80
].copy()

incorrect_predictions.to_csv(
    os.path.join(
        OUT,
        "incorrect_predictions.csv",
    ),
    index=False,
)

high_confidence_errors.to_csv(
    os.path.join(
        OUT,
        "high_confidence_errors.csv",
    ),
    index=False,
)

print("Total predictions:", len(predictions))

print(
    "Correct predictions:",
    int(predictions["correct"].sum()),
)

print(
    "Incorrect predictions:",
    len(incorrect_predictions),
)

print(
    "High-confidence errors:",
    len(high_confidence_errors),
)

print(
    "Abstentions:",
    int(predictions["abstained"].sum()),
)

print("\nMost confident incorrect predictions:")

display(
    incorrect_predictions[
        [
            "dataset",
            "condition",
            "question",
            "reference_answer",
            "predicted_answer",
            "confidence",
            "abstained",
        ]
    ].head(20)
)

In [ ]:
visual_errors = predictions[
    (predictions["dataset"].isin(["SLAKE", "PathVQA"]))
    & (predictions["correct"] == 0)
].copy()

repeated_failures = (
    visual_errors
    .groupby(
        [
            "dataset",
            "item_id",
            "question",
        ]
    )
    .agg(
        failed_conditions=(
            "condition",
            "nunique",
        ),
        conditions=(
            "condition",
            lambda values: ", ".join(
                sorted(set(values))
            ),
        ),
        maximum_confidence=(
            "confidence",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "failed_conditions",
            "maximum_confidence",
        ],
        ascending=False,
    )
)

print("Repeated visual failures:")

display(repeated_failures)

In [ ]:
import os
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"

PRED_PATH = os.path.join(
    OUT,
    "predictions.csv",
)

predictions = pd.read_csv(
    PRED_PATH
)

visual = predictions[
    predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

paired = visual.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
    ],
    columns="condition",
    values="correct",
    aggfunc="first",
).reset_index()

required_conditions = [
    "clean",
    "blur",
    "dark",
]

for condition in required_conditions:
    if condition not in paired.columns:
        paired[condition] = pd.NA

paired["failure_pattern"] = "other"

paired.loc[
    (paired["clean"] == 0)
    & (paired["blur"] == 0)
    & (paired["dark"] == 0),
    "failure_pattern",
] = "persistent_failure"

paired.loc[
    (paired["clean"] == 1)
    & (
        (paired["blur"] == 0)
        | (paired["dark"] == 0)
    ),
    "failure_pattern",
] = "degradation_failure"

paired.loc[
    (paired["clean"] == 0)
    & (
        (paired["blur"] == 1)
        | (paired["dark"] == 1)
    ),
    "failure_pattern",
] = "degradation_recovery"

paired.loc[
    (paired["clean"] == 1)
    & (paired["blur"] == 1)
    & (paired["dark"] == 1),
    "failure_pattern",
] = "consistently_correct"

print("=" * 60)
print("VISUAL FAILURE CLASSIFICATION")
print("=" * 60)

print(
    paired["failure_pattern"]
    .value_counts()
)

print("\nBreakdown by dataset:")

print(
    pd.crosstab(
        paired["dataset"],
        paired["failure_pattern"],
    )
)

paired.to_csv(
    os.path.join(
        OUT,
        "visual_failure_patterns.csv",
    ),
    index=False,
)

print("\nPersistent failures:")

display(
    paired[
        paired["failure_pattern"]
        == "persistent_failure"
    ][
        [
            "dataset",
            "item_id",
            "question",
            "clean",
            "blur",
            "dark",
        ]
    ]
)

print("\nDegradation-induced failures:")

display(
    paired[
        paired["failure_pattern"]
        == "degradation_failure"
    ][
        [
            "dataset",
            "item_id",
            "question",
            "clean",
            "blur",
            "dark",
        ]
    ]
)

In [ ]:
visual_prediction_counts = (
    visual
    .groupby(
        [
            "dataset",
            "condition",
            "predicted_answer",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

visual_prediction_counts["rate"] = (
    visual_prediction_counts["count"]
    / visual_prediction_counts.groupby(
        [
            "dataset",
            "condition",
        ]
    )["count"].transform("sum")
)

print("=" * 60)
print("VISUAL ANSWER DISTRIBUTION")
print("=" * 60)

display(
    visual_prediction_counts
    .sort_values(
        [
            "dataset",
            "condition",
            "predicted_answer",
        ]
    )
)

visual_prediction_counts.to_csv(
    os.path.join(
        OUT,
        "visual_answer_distribution.csv",
    ),
    index=False,
)

print("\nReference answer distribution:")

print(
    visual[
        [
            "dataset",
            "item_id",
            "reference_answer",
        ]
    ]
    .drop_duplicates(
        [
            "dataset",
            "item_id",
        ]
    )
    .groupby(
        [
            "dataset",
            "reference_answer",
        ]
    )
    .size()
)

In [ ]:
import os
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"
PRED_PATH = os.path.join(OUT, "predictions.csv")

if not os.path.isfile(PRED_PATH):
    raise FileNotFoundError(PRED_PATH)

predictions = pd.read_csv(PRED_PATH)

visual = predictions[
    predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

counts = (
    visual
    .groupby(
        [
            "dataset",
            "condition",
            "predicted_answer",
        ]
    )
    .size()
    .reset_index(name="count")
)

totals = counts.groupby(
    [
        "dataset",
        "condition",
    ]
)["count"].transform("sum")

counts["rate"] = counts["count"] / totals
counts["percent"] = counts["rate"] * 100

counts = counts.sort_values(
    [
        "dataset",
        "condition",
        "predicted_answer",
    ]
).reset_index(drop=True)

print("=" * 60)
print("VISUAL ANSWER DISTRIBUTION")
print("=" * 60)

display(counts)

prediction_output = os.path.join(
    OUT,
    "visual_answer_distribution.csv",
)

counts.to_csv(
    prediction_output,
    index=False,
)

reference = (
    visual[
        [
            "dataset",
            "item_id",
            "reference_answer",
        ]
    ]
    .drop_duplicates(
        [
            "dataset",
            "item_id",
        ]
    )
    .groupby(
        [
            "dataset",
            "reference_answer",
        ]
    )
    .size()
    .reset_index(name="count")
)

reference_totals = reference.groupby(
    "dataset"
)["count"].transform("sum")

reference["rate"] = (
    reference["count"]
    / reference_totals
)

reference["percent"] = (
    reference["rate"]
    * 100
)



In [ ]:
import pandas as pd

visual_predictions = final_predictions[
    final_predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

answer_stability = visual_predictions.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
    ],
    columns="condition",
    values="predicted_answer",
    aggfunc="first",
).reset_index()

for condition in [
    "clean",
    "blur",
    "dark",
]:
    if condition not in answer_stability.columns:
        answer_stability[condition] = None

answer_stability["changed_under_blur"] = (
    answer_stability["clean"]
    != answer_stability["blur"]
)

answer_stability["changed_under_darkness"] = (
    answer_stability["clean"]
    != answer_stability["dark"]
)

answer_stability["changed_under_any_degradation"] = (
    answer_stability["changed_under_blur"]
    | answer_stability["changed_under_darkness"]
)

stability_summary = (
    answer_stability
    .groupby("dataset")
    .agg(
        total_items=(
            "item_id",
            "count",
        ),
        changed_under_blur=(
            "changed_under_blur",
            "sum",
        ),
        changed_under_darkness=(
            "changed_under_darkness",
            "sum",
        ),
        changed_under_any_degradation=(
            "changed_under_any_degradation",
            "sum",
        ),
    )
    .reset_index()
)

stability_summary["blur_change_rate"] = (
    stability_summary["changed_under_blur"]
    / stability_summary["total_items"]
)

stability_summary["darkness_change_rate"] = (
    stability_summary["changed_under_darkness"]
    / stability_summary["total_items"]
)

stability_summary["any_change_rate"] = (
    stability_summary[
        "changed_under_any_degradation"
    ]
    / stability_summary["total_items"]
)

print("=" * 60)
print("VISUAL ANSWER STABILITY")
print("=" * 60)

display(stability_summary)

print("\nQuestions with changed answers:")

display(
    answer_stability[
        answer_stability[
            "changed_under_any_degradation"
        ]
    ]
)

answer_stability.to_csv(
    os.path.join(
        CFG["output_directory"],
        "visual_answer_stability.csv",
    ),
    index=False,
)

stability_summary.to_csv(
    os.path.join(
        CFG["output_directory"],
        "visual_stability_summary.csv",
    ),
    index=False,
)

In [ ]:
import pandas as pd

visual_predictions = final_predictions[
    final_predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

answer_stability = visual_predictions.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
    ],
    columns="condition",
    values="predicted_answer",
    aggfunc="first",
).reset_index()

for condition in [
    "clean",
    "blur",
    "dark",
]:
    if condition not in answer_stability.columns:
        answer_stability[condition] = None

answer_stability["changed_under_blur"] = (
    answer_stability["clean"]
    != answer_stability["blur"]
)

answer_stability["changed_under_darkness"] = (
    answer_stability["clean"]
    != answer_stability["dark"]
)

answer_stability["changed_under_any_degradation"] = (
    answer_stability["changed_under_blur"]
    | answer_stability["changed_under_darkness"]
)

stability_summary = (
    answer_stability
    .groupby("dataset")
    .agg(
        total_items=(
            "item_id",
            "count",
        ),
        changed_under_blur=(
            "changed_under_blur",
            "sum",
        ),
        changed_under_darkness=(
            "changed_under_darkness",
            "sum",
        ),
        changed_under_any_degradation=(
            "changed_under_any_degradation",
            "sum",
        ),
    )
    .reset_index()
)

stability_summary["blur_change_rate"] = (
    stability_summary["changed_under_blur"]
    / stability_summary["total_items"]
)

stability_summary["darkness_change_rate"] = (
    stability_summary["changed_under_darkness"]
    / stability_summary["total_items"]
)

stability_summary["any_change_rate"] = (
    stability_summary[
        "changed_under_any_degradation"
    ]
    / stability_summary["total_items"]
)

print("=" * 60)
print("VISUAL ANSWER STABILITY")
print("=" * 60)

display(stability_summary)

print("\nQuestions with changed answers:")

display(
    answer_stability[
        answer_stability[
            "changed_under_any_degradation"
        ]
    ]
)

answer_stability.to_csv(
    os.path.join(
        CFG["output_directory"],
        "visual_answer_stability.csv",
    ),
    index=False,
)

stability_summary.to_csv(
    os.path.join(
        CFG["output_directory"],
        "visual_stability_summary.csv",
    ),
    index=False,
)

In [ ]:
import os
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"

PRED_PATH = os.path.join(
    OUT,
    "predictions.csv",
)

predictions = pd.read_csv(
    PRED_PATH
)

visual = predictions[
    predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

correctness = visual.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
    ],
    columns="condition",
    values="correct",
    aggfunc="first",
).reset_index()

for condition in [
    "clean",
    "blur",
    "dark",
]:
    if condition not in correctness.columns:
        correctness[condition] = pd.NA

correctness["blur_outcome"] = "unchanged"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["blur"] == 0),
    "blur_outcome",
] = "degradation_failure"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["blur"] == 1),
    "blur_outcome",
] = "degradation_recovery"

correctness["dark_outcome"] = "unchanged"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["dark"] == 0),
    "dark_outcome",
] = "degradation_failure"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["dark"] == 1),
    "dark_outcome",
] = "degradation_recovery"

print("=" * 60)
print("DEGRADATION OUTCOMES")
print("=" * 60)

blur_summary = pd.crosstab(
    correctness["dataset"],
    correctness["blur_outcome"],
)

dark_summary = pd.crosstab(
    correctness["dataset"],
    correctness["dark_outcome"],
)

print("Blur outcomes:")
display(blur_summary)

print("Darkness outcomes:")
display(dark_summary)

changed_outcomes = correctness[
    (correctness["blur_outcome"] != "unchanged")
    | (correctness["dark_outcome"] != "unchanged")
].copy()

print("Questions affected by degradation:")
display(
    changed_outcomes[
        [
            "dataset",
            "item_id",
            "question",
            "clean",
            "blur",
            "dark",
            "blur_outcome",
            "dark_outcome",
        ]
    ]
)

OUTPUT_PATH = os.path.join(
    OUT,
    "degradation_outcomes.csv",
)

correctness.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved to:", OUTPUT_PATH)

In [ ]:
condition_results = (
    visual
    .groupby(
        [
            "dataset",
            "condition",
        ]
    )
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        mean_confidence=("confidence", "mean"),
        abstention_rate=("abstained", "mean"),
    )
    .reset_index()
)

condition_results["accuracy_percent"] = (
    condition_results["accuracy"]
    * 100
)

condition_results["confidence_percent"] = (
    condition_results["mean_confidence"]
    * 100
)

print("=" * 60)
print("CONDITION-LEVEL PERFORMANCE")
print("=" * 60)

display(condition_results)

condition_results.to_csv(
    os.path.join(
        OUT,
        "condition_level_performance.csv",
    ),
    index=False,
)

In [ ]:
import os
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"

PRED_PATH = os.path.join(
    OUT,
    "predictions.csv",
)

predictions = pd.read_csv(
    PRED_PATH
)

visual = predictions[
    predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

correctness = visual.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
    ],
    columns="condition",
    values="correct",
    aggfunc="first",
).reset_index()

for condition in [
    "clean",
    "blur",
    "dark",
]:
    if condition not in correctness.columns:
        correctness[condition] = pd.NA

correctness["blur_outcome"] = "unchanged"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["blur"] == 0),
    "blur_outcome",
] = "degradation_failure"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["blur"] == 1),
    "blur_outcome",
] = "degradation_recovery"

correctness["dark_outcome"] = "unchanged"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["dark"] == 0),
    "dark_outcome",
] = "degradation_failure"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["dark"] == 1),
    "dark_outcome",
] = "degradation_recovery"

print("=" * 60)
print("DEGRADATION OUTCOMES")
print("=" * 60)

blur_summary = pd.crosstab(
    correctness["dataset"],
    correctness["blur_outcome"],
)

dark_summary = pd.crosstab(
    correctness["dataset"],
    correctness["dark_outcome"],
)

print("Blur outcomes:")
display(blur_summary)

print("Darkness outcomes:")
display(dark_summary)

changed_outcomes = correctness[
    (correctness["blur_outcome"] != "unchanged")
    | (correctness["dark_outcome"] != "unchanged")
].copy()

print("Questions affected by degradation:")
display(
    changed_outcomes[
        [
            "dataset",
            "item_id",
            "question",
            "clean",
            "blur",
            "dark",
            "blur_outcome",
            "dark_outcome",
        ]
    ]
)

OUTPUT_PATH = os.path.join(
    OUT,
    "degradation_outcomes.csv",
)

correctness.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved to:", OUTPUT_PATH)

In [ ]:
# NEXT ANALYSIS:
# Degradation failures, recoveries, answer bias,
# and paired confidence changes

import os
import numpy as np
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"
PRED_PATH = os.path.join(OUT, "predictions.csv")

if not os.path.isfile(PRED_PATH):
    raise FileNotFoundError(
        "predictions.csv not found at: "
        + PRED_PATH
    )

predictions = pd.read_csv(PRED_PATH)

visual = predictions[
    predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

visual["confidence"] = pd.to_numeric(
    visual["confidence"],
    errors="coerce",
)

visual["correct"] = pd.to_numeric(
    visual["correct"],
    errors="coerce",
)

visual = visual[
    np.isfinite(visual["confidence"])
    & np.isfinite(visual["correct"])
].copy()

print("=" * 60)
print("VISUAL ROBUSTNESS ANALYSIS")
print("=" * 60)

print("Visual prediction rows:", len(visual))
print(
    "Unique visual questions:",
    visual[["dataset", "item_id"]]
    .drop_duplicates()
    .shape[0],
)

# ------------------------------------------------------------
# 1. PAIRED CORRECTNESS TABLE
# ------------------------------------------------------------

correctness = visual.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
        "reference_answer",
    ],
    columns="condition",
    values="correct",
    aggfunc="first",
).reset_index()

for condition in ["clean", "blur", "dark"]:
    if condition not in correctness.columns:
        correctness[condition] = np.nan

correctness["blur_outcome"] = "unchanged"
correctness["dark_outcome"] = "unchanged"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["blur"] == 0),
    "blur_outcome",
] = "degradation_failure"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["blur"] == 1),
    "blur_outcome",
] = "degradation_recovery"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["dark"] == 0),
    "dark_outcome",
] = "degradation_failure"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["dark"] == 1),
    "dark_outcome",
] = "degradation_recovery"

correctness["overall_pattern"] = "mixed_or_stable"

correctness.loc[
    (correctness["clean"] == 1)
    & (correctness["blur"] == 1)
    & (correctness["dark"] == 1),
    "overall_pattern",
] = "consistently_correct"

correctness.loc[
    (correctness["clean"] == 0)
    & (correctness["blur"] == 0)
    & (correctness["dark"] == 0),
    "overall_pattern",
] = "persistent_failure"

print()
print("=" * 60)
print("BLUR OUTCOMES")
print("=" * 60)

blur_summary = pd.crosstab(
    correctness["dataset"],
    correctness["blur_outcome"],
)

display(blur_summary)

print()
print("=" * 60)
print("DARKNESS OUTCOMES")
print("=" * 60)

dark_summary = pd.crosstab(
    correctness["dataset"],
    correctness["dark_outcome"],
)

display(dark_summary)

print()
print("=" * 60)
print("OVERALL FAILURE PATTERNS")
print("=" * 60)

pattern_summary = pd.crosstab(
    correctness["dataset"],
    correctness["overall_pattern"],
)

display(pattern_summary)

# ------------------------------------------------------------
# 2. PREDICTED ANSWER BIAS
# ------------------------------------------------------------

answer_distribution = (
    visual
    .groupby(
        [
            "dataset",
            "condition",
            "predicted_answer",
        ]
    )
    .size()
    .reset_index(name="count")
)

answer_distribution["total"] = (
    answer_distribution
    .groupby(
        [
            "dataset",
            "condition",
        ]
    )["count"]
    .transform("sum")
)

answer_distribution["rate"] = (
    answer_distribution["count"]
    / answer_distribution["total"]
)

answer_distribution["percent"] = (
    answer_distribution["rate"]
    * 100
)

print()
print("=" * 60)
print("PREDICTED ANSWER DISTRIBUTION")
print("=" * 60)

display(
    answer_distribution.sort_values(
        [
            "dataset",
            "condition",
            "predicted_answer",
        ]
    )
)

# ------------------------------------------------------------
# 3. PAIRED CONFIDENCE CHANGES
# ------------------------------------------------------------

confidence_table = visual.pivot_table(
    index=[
        "dataset",
        "item_id",
        "question",
    ],
    columns="condition",
    values="confidence",
    aggfunc="first",
).reset_index()

for condition in ["clean", "blur", "dark"]:
    if condition not in confidence_table.columns:
        confidence_table[condition] = np.nan

confidence_table["blur_confidence_change"] = (
    confidence_table["blur"]
    - confidence_table["clean"]
)

confidence_table["dark_confidence_change"] = (
    confidence_table["dark"]
    - confidence_table["clean"]
)

confidence_summary = (
    confidence_table
    .groupby("dataset")
    .agg(
        items=("item_id", "count"),
        clean_mean_confidence=(
            "clean",
            "mean",
        ),
        blur_mean_confidence=(
            "blur",
            "mean",
        ),
        dark_mean_confidence=(
            "dark",
            "mean",
        ),
        mean_blur_change=(
            "blur_confidence_change",
            "mean",
        ),
        mean_dark_change=(
            "dark_confidence_change",
            "mean",
        ),
    )
    .reset_index()
)

print()
print("=" * 60)
print("PAIRED CONFIDENCE CHANGES")
print("=" * 60)

display(confidence_summary)

# ------------------------------------------------------------
# 4. CONFIDENCE CHANGES ON NEW FAILURES
# ------------------------------------------------------------

failure_confidence = correctness.merge(
    confidence_table[
        [
            "dataset",
            "item_id",
            "blur_confidence_change",
            "dark_confidence_change",
        ]
    ],
    on=[
        "dataset",
        "item_id",
    ],
    how="left",
)

degradation_failures = failure_confidence[
    (failure_confidence["blur_outcome"]
     == "degradation_failure")
    |
    (failure_confidence["dark_outcome"]
     == "degradation_failure")
].copy()

print()
print("=" * 60)
print("DEGRADATION-INDUCED FAILURES")
print("=" * 60)

display(
    degradation_failures[
        [
            "dataset",
            "item_id",
            "question",
            "reference_answer",
            "clean",
            "blur",
            "dark",
            "blur_outcome",
            "dark_outcome",
            "blur_confidence_change",
            "dark_confidence_change",
        ]
    ]
)

# ------------------------------------------------------------
# 5. SAVE ALL ANALYSIS FILES
# ------------------------------------------------------------

correctness.to_csv(
    os.path.join(
        OUT,
        "degradation_outcomes.csv",
    ),
    index=False,
)

answer_distribution.to_csv(
    os.path.join(
        OUT,
        "visual_answer_distribution.csv",
    ),
    index=False,
)

confidence_table.to_csv(
    os.path.join(
        OUT,
        "paired_confidence_changes.csv",
    ),
    index=False,
)

confidence_summary.to_csv(
    os.path.join(
        OUT,
        "confidence_change_summary.csv",
    ),
    index=False,
)

degradation_failures.to_csv(
    os.path.join(
        OUT,
        "degradation_induced_failures.csv",
    ),
    index=False,
)

print()
print("=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)

print(
    "Degradation failures:",
    len(degradation_failures),
)

print(
    "Persistent failures:",
    int(
        (
            correctness["overall_pattern"]
            == "persistent_failure"
        ).sum()
    ),
)

print(
    "Consistently correct:",
    int(
        (
            correctness["overall_pattern"]
            == "consistently_correct"
        ).sum()
    ),
)

print("Files saved to:", OUT)

In [ ]:
import os
import math
import numpy as np
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"
PRED_PATH = os.path.join(OUT, "predictions.csv")

data = pd.read_csv(PRED_PATH)

visual = data[
    data["dataset"].isin(["PathVQA", "SLAKE"])
].copy()

visual["correct"] = pd.to_numeric(
    visual["correct"],
    errors="coerce"
)

visual["confidence"] = pd.to_numeric(
    visual["confidence"],
    errors="coerce"
)


def exact_p(a, b):
    total = a + b

    if total == 0:
        return np.nan

    smaller = min(a, b)

    probability = 0.0

    for value in range(total + 1):
        if value <= smaller or value >= total - smaller:
            probability += (
                math.comb(total, value)
                * 0.5 ** total
            )

    return min(1.0, probability)


rows = []
detail_frames = []

for dataset_name in ["PathVQA", "SLAKE"]:

    dataset = visual[
        visual["dataset"] == dataset_name
    ].copy()

    clean = dataset[
        dataset["condition"] == "clean"
    ][
        [
            "item_id",
            "question",
            "reference_answer",
            "predicted_answer",
            "correct",
            "confidence",
        ]
    ].copy()

    clean = clean.rename(
        columns={
            "predicted_answer": "clean_answer",
            "correct": "clean_correct",
            "confidence": "clean_confidence",
        }
    )

    for condition in ["blur", "dark"]:

        degraded = dataset[
            dataset["condition"] == condition
        ][
            [
                "item_id",
                "predicted_answer",
                "correct",
                "confidence",
            ]
        ].copy()

        degraded = degraded.rename(
            columns={
                "predicted_answer": "degraded_answer",
                "correct": "degraded_correct",
                "confidence": "degraded_confidence",
            }
        )

        paired = clean.merge(
            degraded,
            on="item_id",
            how="inner"
        )

        paired["dataset"] = dataset_name
        paired["comparison"] = "clean_vs_" + condition

        paired["new_failure"] = (
            (paired["clean_correct"] == 1)
            & (paired["degraded_correct"] == 0)
        )

        paired["recovery"] = (
            (paired["clean_correct"] == 0)
            & (paired["degraded_correct"] == 1)
        )

        paired["answer_changed"] = (
            paired["clean_answer"]
            != paired["degraded_answer"]
        )

        paired["yes_to_no"] = (
            (paired["clean_answer"] == "Yes")
            & (paired["degraded_answer"] == "No")
        )

        paired["no_to_yes"] = (
            (paired["clean_answer"] == "No")
            & (paired["degraded_answer"] == "Yes")
        )

        paired["confidence_change"] = (
            paired["degraded_confidence"]
            - paired["clean_confidence"]
        )

        failures = int(
            paired["new_failure"].sum()
        )

        recoveries = int(
            paired["recovery"].sum()
        )

        yes_to_no = int(
            paired["yes_to_no"].sum()
        )

        no_to_yes = int(
            paired["no_to_yes"].sum()
        )

        rows.append({
            "dataset": dataset_name,
            "comparison": "clean_vs_" + condition,
            "n": len(paired),
            "clean_accuracy": paired[
                "clean_correct"
            ].mean(),
            "degraded_accuracy": paired[
                "degraded_correct"
            ].mean(),
            "accuracy_change": (
                paired["degraded_correct"].mean()
                - paired["clean_correct"].mean()
            ),
            "answer_changes": int(
                paired["answer_changed"].sum()
            ),
            "new_failures": failures,
            "recoveries": recoveries,
            "mcnemar_exact_p": exact_p(
                failures,
                recoveries
            ),
            "yes_to_no": yes_to_no,
            "no_to_yes": no_to_yes,
            "answer_shift_p": exact_p(
                yes_to_no,
                no_to_yes
            ),
            "mean_confidence_change": paired[
                "confidence_change"
            ].mean(),
        })

        detail_frames.append(paired)


statistics = pd.DataFrame(rows)

paired_details = pd.concat(
    detail_frames,
    ignore_index=True
)

print("=" * 60)
print("PAIRED ROBUSTNESS STATISTICS")
print("=" * 60)

display(
    statistics.round(4)
)


new_failures = paired_details[
    paired_details["new_failure"]
].copy()

recoveries = paired_details[
    paired_details["recovery"]
].copy()


print()
print("=" * 60)
print("NEW DEGRADATION FAILURES")
print("=" * 60)

display(
    new_failures[
        [
            "dataset",
            "comparison",
            "item_id",
            "question",
            "reference_answer",
            "clean_answer",
            "degraded_answer",
            "clean_confidence",
            "degraded_confidence",
            "confidence_change",
        ]
    ]
)


print()
print("=" * 60)
print("DEGRADATION RECOVERIES")
print("=" * 60)

display(
    recoveries[
        [
            "dataset",
            "comparison",
            "item_id",
            "question",
            "reference_answer",
            "clean_answer",
            "degraded_answer",
            "clean_confidence",
            "degraded_confidence",
            "confidence_change",
        ]
    ]
)


statistics.to_csv(
    os.path.join(
        OUT,
        "paired_robustness_statistics.csv"
    ),
    index=False
)

paired_details.to_csv(
    os.path.join(
        OUT,
        "paired_robustness_items.csv"
    ),
    index=False
)

new_failures.to_csv(
    os.path.join(
        OUT,
        "degradation_failure_details.csv"
    ),
    index=False
)

recoveries.to_csv(
    os.path.join(
        OUT,
        "degradation_recovery_details.csv"
    ),
    index=False
)


print()
print("=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)

print("New degradation failures:", len(new_failures))
print("Degradation recoveries:", len(recoveries))
print("Files saved to:", OUT)

In [ ]:
import os
import numpy as np
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"
PRED_PATH = os.path.join(OUT, "predictions.csv")

predictions = pd.read_csv(PRED_PATH)

predictions["correct"] = pd.to_numeric(
    predictions["correct"],
    errors="coerce",
)

predictions["confidence"] = pd.to_numeric(
    predictions["confidence"],
    errors="coerce",
)

predictions = predictions[
    np.isfinite(predictions["correct"])
    & np.isfinite(predictions["confidence"])
].copy()

afrimed = predictions[
    predictions["dataset"] == "AfriMed-QA"
].copy()

if afrimed.empty:
    raise ValueError(
        "No AfriMed-QA predictions were found."
    )

afrimed["confidence_bin"] = pd.cut(
    afrimed["confidence"],
    bins=[
        0.0,
        0.4,
        0.5,
        0.6,
        0.7,
        0.8,
        0.9,
        1.0,
    ],
    include_lowest=True,
)

calibration_bins = (
    afrimed
    .groupby(
        "confidence_bin",
        observed=True,
    )
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        mean_confidence=(
            "confidence",
            "mean",
        ),
    )
    .reset_index()
)

calibration_bins["calibration_gap"] = (
    calibration_bins["mean_confidence"]
    - calibration_bins["accuracy"]
)

incorrect = afrimed[
    afrimed["correct"] == 0
].copy()

incorrect = incorrect.sort_values(
    "confidence",
    ascending=False,
)

high_confidence_errors = incorrect[
    incorrect["confidence"] >= 0.80
].copy()

print("=" * 60)
print("AFRIMED-QA BASELINE ANALYSIS")
print("=" * 60)

print("Total questions:", len(afrimed))

print(
    "Correct answers:",
    int(afrimed["correct"].sum()),
)

print(
    "Incorrect answers:",
    len(incorrect),
)

print(
    "Accuracy:",
    round(
        afrimed["correct"].mean(),
        4,
    ),
)

print(
    "Mean confidence:",
    round(
        afrimed["confidence"].mean(),
        4,
    ),
)

print(
    "Confidence minus accuracy:",
    round(
        afrimed["confidence"].mean()
        - afrimed["correct"].mean(),
        4,
    ),
)

print(
    "High-confidence errors:",
    len(high_confidence_errors),
)

print()
print("=" * 60)
print("CALIBRATION BY CONFIDENCE BIN")
print("=" * 60)

display(
    calibration_bins.round(4)
)

print()
print("=" * 60)
print("MOST CONFIDENT AFRIMED-QA ERRORS")
print("=" * 60)

display(
    incorrect[
        [
            "item_id",
            "domain",
            "country",
            "question",
            "reference_answer",
            "predicted_answer",
            "confidence",
        ]
    ].head(15)
)

print()
print("=" * 60)
print("HIGH-CONFIDENCE AFRIMED-QA ERRORS")
print("=" * 60)

display(
    high_confidence_errors[
        [
            "item_id",
            "domain",
            "country",
            "question",
            "reference_answer",
            "predicted_answer",
            "confidence",
        ]
    ]
)

calibration_bins.to_csv(
    os.path.join(
        OUT,
        "afrimed_calibration_bins.csv",
    ),
    index=False,
)

incorrect.to_csv(
    os.path.join(
        OUT,
        "afrimed_incorrect_predictions.csv",
    ),
    index=False,
)

high_confidence_errors.to_csv(
    os.path.join(
        OUT,
        "afrimed_high_confidence_errors.csv",
    ),
    index=False,
)

print()
print("=" * 60)
print("AFRIMED-QA ANALYSIS COMPLETE")
print("=" * 60)

print(
    "Files saved to:",
    OUT,
)

In [ ]:
import os
import numpy as np
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"
PATH = os.path.join(OUT, "predictions.csv")

data = pd.read_csv(PATH)

afrimed = data[
    data["dataset"] == "AfriMed-QA"
].copy()

afrimed["correct"] = pd.to_numeric(
    afrimed["correct"],
    errors="coerce"
)

afrimed["confidence"] = pd.to_numeric(
    afrimed["confidence"],
    errors="coerce"
)

afrimed = afrimed.dropna(
    subset=["correct", "confidence"]
)

correct = afrimed["correct"].to_numpy(float)
confidence = afrimed["confidence"].to_numpy(float)

n = len(afrimed)
bootstrap_samples = 10000
rng = np.random.default_rng(42)

indices = rng.integers(
    0,
    n,
    size=(bootstrap_samples, n)
)

sampled_correct = correct[indices]
sampled_confidence = confidence[indices]

boot_accuracy = sampled_correct.mean(axis=1)

boot_confidence = sampled_confidence.mean(axis=1)

boot_gap = (
    boot_confidence
    - boot_accuracy
)

boot_high_error = (
    (
        (sampled_correct == 0)
        & (sampled_confidence >= 0.80)
    )
    .mean(axis=1)
)

def interval(values):
    return np.percentile(
        values,
        [2.5, 97.5]
    )

rows = []

metrics = [
    (
        "accuracy",
        correct.mean(),
        boot_accuracy
    ),
    (
        "mean_confidence",
        confidence.mean(),
        boot_confidence
    ),
    (
        "confidence_minus_accuracy",
        confidence.mean() - correct.mean(),
        boot_gap
    ),
    (
        "high_confidence_error_rate",
        (
            (correct == 0)
            & (confidence >= 0.80)
        ).mean(),
        boot_high_error
    ),
]

for name, estimate, values in metrics:
    lower, upper = interval(values)

    rows.append({
        "metric": name,
        "estimate": estimate,
        "ci_lower": lower,
        "ci_upper": upper,
        "estimate_percent": estimate * 100,
        "ci_lower_percent": lower * 100,
        "ci_upper_percent": upper * 100,
    })

bootstrap_summary = pd.DataFrame(rows)

print("=" * 60)
print("AFRIMED-QA BOOTSTRAP INTERVALS")
print("=" * 60)

display(
    bootstrap_summary.round(4)
)

output_path = os.path.join(
    OUT,
    "afrimed_bootstrap_intervals.csv"
)

bootstrap_summary.to_csv(
    output_path,
    index=False
)

gap_row = bootstrap_summary[
    bootstrap_summary["metric"]
    == "confidence_minus_accuracy"
].iloc[0]

print()
print("Questions:", n)
print("Bootstrap samples:", bootstrap_samples)
print("Saved to:", output_path)

if gap_row["ci_lower"] > 0:
    print()
    print(
        "The overconfidence interval is entirely above zero."
    )
    print(
        "This supports a reproducible overconfidence finding."
    )
else:
    print()
    print(
        "The overconfidence interval includes zero."
    )

print()
print("Bootstrap analysis completed successfully.")

In [ ]:
import os
import math
import numpy as np
import pandas as pd

OUT = "/kaggle/working/healthcare_audit_baseline_160"
PRED_PATH = os.path.join(OUT, "predictions.csv")

predictions = pd.read_csv(PRED_PATH)

visual = predictions[
    predictions["dataset"].isin(
        ["SLAKE", "PathVQA"]
    )
].copy()

visual["correct"] = pd.to_numeric(
    visual["correct"],
    errors="coerce",
)

visual["confidence"] = pd.to_numeric(
    visual["confidence"],
    errors="coerce",
)

visual = visual[
    np.isfinite(visual["correct"])
    & np.isfinite(visual["confidence"])
].copy()


def exact_binomial_two_sided(
    successes,
    trials,
):
    """
    Exact two-sided binomial test under p = 0.5.

    Used for paired discordant outcomes and
    directional answer changes.
    """

    if trials == 0:
        return np.nan

    observed_probability = (
        math.comb(trials, successes)
        * (0.5 ** trials)
    )

    probability = 0.0

    for value in range(trials + 1):
        value_probability = (
            math.comb(trials, value)
            * (0.5 ** trials)
        )

        if (
            value_probability
            <= observed_probability + 1e-15
        ):
            probability += value_probability

    return min(
        1.0,
        float(probability),
    )


def analyze_condition(
    dataset_frame,
    degraded_condition,
):
    """
    Compare clean versus one degraded condition.
    """

    clean = dataset_frame[
        dataset_frame["condition"] == "clean"
    ][
        [
            "item_id",
            "question",
            "reference_answer",
            "predicted_answer",
            "correct",
            "confidence",
        ]
    ].copy()

    clean = clean.rename(
        columns={
            "predicted_answer": (
                "clean_prediction"
            ),
            "correct": "clean_correct",
            "confidence": (
                "clean_confidence"
            ),
        }
    )

    degraded = dataset_frame[
        dataset_frame["condition"]
        == degraded_condition
    ][
        [
            "item_id",
            "predicted_answer",
            "correct",
            "confidence",
        ]
    ].copy()

    degraded = degraded.rename(
        columns={
            "predicted_answer": (
                "degraded_prediction"
            ),
            "correct": (
                "degraded_correct"
            ),
            "confidence": (
                "degraded_confidence"
            ),
        }
    )

    paired = clean.merge(
        degraded,
        on="item_id",
        how="inner",
    )

    paired["confidence_change"] = (
        paired["degraded_confidence"]
        - paired["clean_confidence"]
    )

    paired["answer_changed"] = (
        paired["clean_prediction"]
        != paired["degraded_prediction"]
    )

    paired["new_failure"] = (
        (paired["clean_correct"] == 1)
        & (paired["degraded_correct"] == 0)
    )

    paired["recovery"] = (
        (paired["clean_correct"] == 0)
        & (paired["degraded_correct"] == 1)
    )

    paired["yes_to_no"] = (
        (paired["clean_prediction"] == "Yes")
        & (
            paired["degraded_prediction"]
            == "No"
        )
    )

    paired["no_to_yes"] = (
        (paired["clean_prediction"] == "No")
        & (
            paired["degraded_prediction"]
            == "Yes"
        )
    )

    new_failures = int(
        paired["new_failure"].sum()
    )

    recoveries = int(
        paired["recovery"].sum()
    )

    discordant_correctness = (
        new_failures + recoveries
    )

    mcnemar_p = exact_binomial_two_sided(
        min(
            new_failures,
            recoveries,
        ),
        discordant_correctness,
    )

    yes_to_no = int(
        paired["yes_to_no"].sum()
    )

    no_to_yes = int(
        paired["no_to_yes"].sum()
    )

    directional_changes = (
        yes_to_no + no_to_yes
    )

    answer_shift_p = (
        exact_binomial_two_sided(
            min(
                yes_to_no,
                no_to_yes,
            ),
            directional_changes,
        )
    )

    failure_rows = paired[
        paired["new_failure"]
    ]

    recovery_rows = paired[
        paired["recovery"]
    ]

    result = {
        "dataset": dataset_frame[
            "dataset"
        ].iloc[0],

        "comparison": (
            "clean_vs_"
            + degraded_condition
        ),

        "n_items": len(paired),

        "clean_accuracy": float(
            paired[
                "clean_correct"
            ].mean()
        ),

        "degraded_accuracy": float(
            paired[
                "degraded_correct"
            ].mean()
        ),

        "accuracy_change": float(
            paired[
                "degraded_correct"
            ].mean()
            - paired[
                "clean_correct"
            ].mean()
        ),

        "answer_changes": int(
            paired[
                "answer_changed"
            ].sum()
        ),

        "new_failures": new_failures,

        "recoveries": recoveries,

        "mcnemar_exact_p": mcnemar_p,

        "yes_to_no_changes": yes_to_no,

        "no_to_yes_changes": no_to_yes,

        "answer_shift_exact_p": (
            answer_shift_p
        ),

        "mean_confidence_change": float(
            paired[
                "confidence_change"
            ].mean()
        ),

        "mean_failure_confidence_change": (
            float(
                failure_rows[
                    "confidence_change"
                ].mean()
            )
            if not failure_rows.empty
            else np.nan
        ),

        "mean_recovery_confidence_change": (
            float(
                recovery_rows[
                    "confidence_change"
                ].mean()
            )
            if not recovery_rows.empty
            else np.nan
        ),
    }

    return result, paired


summary_rows = []
paired_outputs = []

for dataset_name in [
    "PathVQA",
    "SLAKE",
]:
    dataset_frame = visual[
        visual["dataset"] == dataset_name
    ].copy()

    for condition in [
        "blur",
        "dark",
    ]:
        result, paired = analyze_condition(
            dataset_frame,
            condition,
        )

        summary_rows.append(result)

        paired["dataset"] = dataset_name
        paired["comparison"] = (
            "clean_vs_" + condition
        )

        paired_outputs.append(paired)


statistical_summary = pd.DataFrame(
    summary_rows
)

paired_results = pd.concat(
    paired_outputs,
    ignore_index=True,
)


print("=" * 60)
print("PAIRED ROBUSTNESS STATISTICS")
print("=" * 60)

display(
    statistical_summary.round(4)
)


print()
print("=" * 60)
print("NEW DEGRADATION FAILURES")
print("=" * 60)

new_failure_details = paired_results[
    paired_results["new_failure"]
].copy()

display(
    new_failure_details[
        [
            "dataset",
            "comparison",
            "item_id",
            "question",
            "reference_answer",
            "clean_prediction",
            "degraded_prediction",
            "clean_confidence",
            "degraded_confidence",
            "confidence_change",
        ]
    ]
)


print()
print("=" * 60)
print("DEGRADATION RECOVERIES")
print("=" * 60)

recovery_details = paired_results[
    paired_results["recovery"]
].copy()

display(
    recovery_details[
        [
            "dataset",
            "comparison",
            "item_id",
            "question",
            "reference_answer",
            "clean_prediction",
            "degraded_prediction",
            "clean_confidence",
            "degraded_confidence",
            "confidence_change",
        ]
    ]
)


SUMMARY_PATH = os.path.join(
    OUT,
    "paired_robustness_statistics.csv",
)

PAIRED_PATH = os.path.join(
    OUT,
    "paired_robustness_items.csv",
)

FAILURE_PATH = os.path.join(
    OUT,
    "degradation_failure_details.csv",
)

RECOVERY_PATH = os.path.join(
    OUT,
    "degradation_recovery_details.csv",
)


statistical_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

paired_results.to_csv(
    PAIRED_PATH,
    index=False,
)

new_failure_details.to_csv(
    FAILURE_PATH,
    index=False,
)

recovery_details.to_csv(
    RECOVERY_PATH,
    index=False,
)


print()
print("=" * 60)
print("STATISTICAL ANALYSIS COMPLETE")
print("=" * 60)

print(
    "Statistical summary:",
    SUMMARY_PATH,
)

print(
    "Paired item results:",
    PAIRED_PATH,
)

print(
    "Failure details:",
    FAILURE_PATH,
)

print(
    "Recovery details:",
    RECOVERY_PATH,
)